In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
# ART library se zaroori imports
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump

print("Sabhi libraries aache se import ho gayi hain! ✅")

c:\Users\YASH VIJAY SINGH\OneDrive\Desktop\mini-project sem-v\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\YASH VIJAY SINGH\OneDrive\Desktop\mini-project sem-v\.venv\Lib\site-packages\art\estimators\certification\__init__.py:30: UserWarning: PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality
  warnings.warn("PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality")


Sabhi libraries aache se import ho gayi hain! ✅


In [12]:
# New cell (run this right after loading the data)
print("Checking the diversity of the loaded sample:")
print(df['Attack Type'].value_counts())

Checking the diversity of the loaded sample:
Attack Type
Normal Traffic    97385
Port Scanning     82615
Name: count, dtype: int64


In [3]:
# Cell 2: Dataset ka Ek Sample Load Karna 💾
try:
    # We are adding nrows=300000 to load only the first 300,000 rows
    df = pd.read_csv('./data/raw_dataset/cicids2017.csv', nrows=180000)    
    print(f"Successfully loaded a sample of {df.shape[0]} rows. ✅")
    display(df.head())
except FileNotFoundError:
    print(f"❌ ERROR: File not found. Please check the path.")

Successfully loaded a sample of 180000 rows. ✅


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [4]:
print(df.columns)

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Length of Fwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length',
       'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length',
       'Max Packet Length', 'Packet Length Mean', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count',
       'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes',
       'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_p

In [13]:
# Cell 3: Features (X) aur Label (y) ko Alag Karna 🎯

# We use the correct column name: 'Attack Type'
correct_label_name = 'Attack Type' 

X = df.drop(columns=[correct_label_name])
y = df[correct_label_name]
print("Features (X) and Label (y) have been separated successfully. ✅")

Features (X) and Label (y) have been separated successfully. ✅


In [14]:
# Cell 4: Convert Labels to Binary (0 for Normal, 1 for Attack)

# We use 'Normal Traffic' as the condition for class 0.
y_binary = y.apply(lambda x: 0 if x == 'Normal Traffic' else 1)

print("Labels have been converted to a binary format. ✅")

# Check the new distribution of 0s and 1s
print("\nNew binary label distribution:")
print(y_binary.value_counts())

Labels have been converted to a binary format. ✅

New binary label distribution:
Attack Type
0    97385
1    82615
Name: count, dtype: int64


In [16]:
# Cell 5: Data ko Training aur Testing me Baantna (Split)
from sklearn.model_selection import train_test_split

# We use the features 'X' and our new 'y_binary' labels.
# 'stratify=y_binary' ensures both sets have a similar ratio of attacks to normal traffic.
X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.3, random_state=42, stratify=y_binary)

print("Data successfully split. ✅")
print(f"Training data has {X_train.shape[0]} samples.")
print(f"Testing data has {X_test.shape[0]} samples.")

Data successfully split. ✅
Training data has 126000 samples.
Testing data has 54000 samples.


In [17]:
# Cell 6: Feature Scaling
from sklearn.preprocessing import MinMaxScaler

# Create a scaler object
scaler = MinMaxScaler()

# IMPORTANT: We learn the scaling rules from the training data ONLY.
X_train_scaled = scaler.fit_transform(X_train)

# Then, we apply those same rules to the test data.
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully. ✅")
print("Your training and testing data are now ready for the model.")

Feature scaling completed successfully. ✅
Your training and testing data are now ready for the model.


In [18]:
# Cell 7: Train the IDS Model
from sklearn.ensemble import RandomForestClassifier

# We will use a RandomForestClassifier.
# n_jobs=-1 tells the model to use all available CPU cores to speed up training.
ids_model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)

print("Model training is starting... (This may take a moment)")
# Train the model on the scaled training data.
ids_model.fit(X_train_scaled, y_train)
print("Model training complete! ✅")

Model training is starting... (This may take a moment)
Model training complete! ✅


In [19]:
# Cell 8: Evaluate Model Performance (Baseline)
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating the model on the unseen test data...")

# Make predictions on the scaled test data
predictions = ids_model.predict(X_test_scaled)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"\nBaseline Model Accuracy: {accuracy * 100:.2f}%")

# Print a detailed report (precision, recall, f1-score)
print("\nClassification Report:\n")
print(classification_report(y_test, predictions, target_names=['Normal Traffic (0)', 'Attack (1)']))


Evaluating the model on the unseen test data...

Baseline Model Accuracy: 99.99%

Classification Report:

                    precision    recall  f1-score   support

Normal Traffic (0)       1.00      1.00      1.00     29216
        Attack (1)       1.00      1.00      1.00     24784

          accuracy                           1.00     54000
         macro avg       1.00      1.00      1.00     54000
      weighted avg       1.00      1.00      1.00     54000



In [20]:
# Cell 9: Wrap the Model for the ART Library
from art.estimators.classification import SklearnClassifier

# Wrap the trained ids_model in an ART classifier.
# We must specify clip_values=(0, 1) because our data was scaled to that range.
art_classifier = SklearnClassifier(model=ids_model, clip_values=(0, 1))

print("Model has been successfully wrapped for ART. ✅")
print("Ready to select samples for the attack.")

Model has been successfully wrapped for ART. ✅
Ready to select samples for the attack.


In [21]:
import numpy as np

print(f"Unique values in y_test (True Labels): {np.unique(y_test)}")
print(f"Unique values in predictions: {np.unique(predictions)}")

Unique values in y_test (True Labels): [0 1]
Unique values in predictions: [0 1]


In [23]:
# Cell 10: Select Samples for the Attack 🎯
import numpy as np

# We only want to attack samples that are actually malicious (label = 1).
# First, find the indices of all attack samples in the test set.
attack_indices = np.where(y_test.to_numpy() == 1)[0]

# Use these indices to get the corresponding data from X_test_scaled.
x_test_attacks = X_test_scaled[attack_indices]

# The attack can be slow, so we'll start by attacking only the first 50 samples.
# You can increase this number later if you want.
n_samples_to_attack = 50
x_to_attack = x_test_attacks[:n_samples_to_attack]

print(f"Selected {len(x_to_attack)} attack samples to modify. ✅")

Selected 50 attack samples to modify. ✅


In [25]:
# Cell 11: Generate Adversarial Examples (The Attack!)
from art.attacks.evasion import HopSkipJump

# Initialize the HopSkipJump attack.
attack = HopSkipJump(classifier=art_classifier, targeted=False, max_iter=20, max_eval=1000)

print("Generating adversarial examples... This will take several minutes! ☕")
# Generate the adversarial examples from the selected attack samples.
# Corrected the variable name to x_to_attack
x_test_adversarial = attack.generate(x=x_to_attack)

print("Adversarial examples generated successfully! ✅")

Generating adversarial examples... This will take several minutes! ☕


HopSkipJump: 100%|██████████| 50/50 [04:28<00:00,  5.36s/it]

Adversarial examples generated successfully! ✅


In [26]:
# Cell 12: Evaluate the Attack's Success
from sklearn.metrics import accuracy_score
import numpy as np

print("--- Attack Evaluation ---")

# Use the original model to make predictions on the new adversarial samples.
adversarial_predictions = ids_model.predict(x_test_adversarial)

# The true labels for all these samples are '1' (Attack).
original_labels = np.ones(len(x_test_adversarial))

# Check the model's accuracy on this adversarial data. It should be very low.
adv_accuracy = accuracy_score(original_labels, adversarial_predictions)
print(f"Model Accuracy on Adversarial Samples: {adv_accuracy * 100:.2f}%")

# The most important metric: Attack Success Rate.
# This is the percentage of attacks that were successfully misclassified as 'Normal Traffic' (label 0).
misclassified_as_normal = np.sum(adversarial_predictions == 0)
total_attacked = len(x_test_adversarial)
attack_success_rate = (misclassified_as_normal / total_attacked) * 100

print(f"\nTotal attack samples generated: {total_attacked}")
print(f"Samples misclassified as 'Normal Traffic': {misclassified_as_normal}")
print(f"🏆 Attack Success Rate: {attack_success_rate:.2f}%")

--- Attack Evaluation ---
Model Accuracy on Adversarial Samples: 0.00%

Total attack samples generated: 50
Samples misclassified as 'Normal Traffic': 50
🏆 Attack Success Rate: 100.00%


In [27]:
# This is the code from your last cell that calculates the accuracy

from sklearn.metrics import accuracy_score
import numpy as np

# 1. The model makes predictions on the disguised, adversarial samples.
#    Since the attack was 100% successful, this variable likely contains all 0s.
adversarial_predictions = ids_model.predict(x_test_adversarial)

# 2. We create the "answer key". We know all of these samples were
#    originally attacks, so their true labels are all 1s.
original_labels = np.ones(len(x_test_adversarial))

# 3. The accuracy_score function compares the predictions with the answer key.
#    It's comparing a list of all 0s (predictions) with a list of all 1s (true labels).
#    Since none of them match, the score is 0.
adv_accuracy = accuracy_score(original_labels, adversarial_predictions)

# 4. This line prints the final result, formatted as a percentage.
print(f"Model Accuracy on Adversarial Samples: {adv_accuracy * 100:.2f}%")


Model Accuracy on Adversarial Samples: 0.00%


In [28]:
import joblib

# Define a filename for your model
filename = 'ids_model.joblib'

# Save the model to the file
joblib.dump(ids_model, filename)

print(f"Model successfully saved to '{filename}'! ✅")

Model successfully saved to 'ids_model.joblib'! ✅
